In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from torch.utils.data import DataLoader, TensorDataset

##############################################################
# Load data
##############################################################

df = pd.read_csv("data.csv")

X = df[["COM_x", "COM_y", "Var_x", "Var_y", "Cov_xy"]].values

Y = df[["True_x", "True_y"]].values

##############################################################
# Residual targets
##############################################################

residuals = np.zeros_like(Y)

residuals[:,0] = Y[:,0] - X[:,0]
residuals[:,1] = Y[:,1] - X[:,1]

##############################################################
# Normalize inputs
##############################################################

scaler = StandardScaler()
X = scaler.fit_transform(X)

##############################################################
# Train/Test split
##############################################################

X_train, X_test, y_train, y_test = train_test_split(
    X,
    residuals,
    test_size=0.2,
    random_state=42
)

##############################################################
# Convert to tensors
##############################################################

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

y_train = torch.FloatTensor(y_train)
y_test = torch.FloatTensor(y_test)

##############################################################
# DataLoader
##############################################################

train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

##############################################################
# Network
##############################################################

class PositionMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(

            nn.Linear(5, 32),
            nn.SiLU(),

            nn.Linear(32, 32),
            nn.SiLU(),

            nn.Linear(32, 16),
            nn.SiLU(),

            nn.Linear(16, 2)

        )

    def forward(self, x):
        return self.model(x)

##############################################################
# Initialize
##############################################################

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PositionMLP().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

##############################################################
# Training
##############################################################

epochs = 200

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        pred = model(xb)

        loss = criterion(pred, yb)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch+1}/{epochs} "
            f"Loss = {running_loss/len(train_loader):.6f}"
        )

##############################################################
# Evaluation
##############################################################

model.eval()

with torch.no_grad():

    pred_residual = model(X_test.to(device)).cpu().numpy()

##############################################################
# Recover corrected positions
##############################################################

# Recover original COM values
X_test_original = scaler.inverse_transform(X_test.numpy())

pred_position = np.zeros((len(pred_residual),2))

pred_position[:,0] = (
    X_test_original[:,0] +
    pred_residual[:,0]
)

pred_position[:,1] = (
    X_test_original[:,1] +
    pred_residual[:,1]
)

##############################################################
# True positions
##############################################################

true_position = np.zeros_like(pred_position)

true_position[:,0] = (
    X_test_original[:,0] +
    y_test.numpy()[:,0]
)

true_position[:,1] = (
    X_test_original[:,1] +
    y_test.numpy()[:,1]
)

##############################################################
# Metrics
##############################################################

rmse = np.sqrt(
    mean_squared_error(
        true_position,
        pred_position
    )
)

print(f"\nRMSE = {rmse:.4f}")

##############################################################
# Example predictions
##############################################################

print("\nFirst 10 predictions:\n")

for i in range(10):

    print(
        f"Pred: ({pred_position[i,0]:.3f}, "
        f"{pred_position[i,1]:.3f})   "
        f"True: ({true_position[i,0]:.3f}, "
        f"{true_position[i,1]:.3f})"
    )

##############################################################
# Save model
##############################################################

torch.save(model.state_dict(), "position_mlp.pt")

##############################################################
# Save scaler
##############################################################

import joblib

joblib.dump(scaler, "scaler.pkl")